# Integrated Skin Condition Classification and Treatment Recommendation System

This notebook integrates:
1. CNN-based skin condition classification model (EfficientNetV2M)
2. RAG pipeline with hybrid search for treatment recommendations

Workflow:
- User uploads an image of a skin condition
- CNN model classifies the condition
- RAG pipeline retrieves treatment guidelines based on the classification

## 1. Install Required Dependencies

In [ ]:
# Install packages for RAG pipeline
%pip install rank-bm25
%pip install langchain
%pip install langchain-core
%pip install langchain-community
%pip install langchain-google-genai
%pip install langchain-openai
%pip install langchain-classic chromadb
%pip install huggingface-hub

## 2. Import Libraries

In [ ]:
# CNN Model imports
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input
from PIL import Image

# RAG Pipeline imports
import getpass
from pathlib import Path
from typing import List, Dict, Optional

from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document

print("All libraries imported successfully")

## 3. Mount Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted successfully")

## 4. Configuration

In [ ]:
# Google Drive paths - Update these to match your Drive structure
DRIVE_BASE_PATH = "/content/drive/MyDrive"  # Base path to your Google Drive

# CNN Model Configuration
MODEL_PATH = f"{DRIVE_BASE_PATH}/best_model.keras"  # Path to your trained model in Drive
IMG_SIZE = (224, 224)  # Image size for EfficientNetV2M

# Skin condition classes (update based on your model)
CLASS_NAMES = [
    "Acne",
    "Eczema",
    "Psoriasis",
    "Rosacea",
    "Urticaria",
    "Vitiligo"
]

# RAG Pipeline Configuration
VECTOR_STORE_PATH = f"{DRIVE_BASE_PATH}/chroma_db"  # Path to chroma_db in Drive
CHUNKING_METHOD = "recursive"  # Options: "recursive" or "agentic"
HYBRID_WEIGHTS = [0.4, 0.6]  # 40% BM25, 60% Semantic
RETRIEVAL_K = 5

print("Configuration loaded")
print(f"Model path: {MODEL_PATH}")
print(f"Vector store path: {VECTOR_STORE_PATH}")
print(f"Image size: {IMG_SIZE}")
print(f"Number of classes: {len(CLASS_NAMES)}")
print(f"Classes: {CLASS_NAMES}")

## 5. Setup API Keys

In [ ]:
# Google API Key for Gemini Pro
if "GOOGLE_API_KEY" not in os.environ or not os.environ["GOOGLE_API_KEY"]:
    api_key = getpass.getpass("Enter your Google API Key: ")
    if not api_key or not api_key.strip():
        raise ValueError("Google API key is required")
    os.environ["GOOGLE_API_KEY"] = api_key.strip()

print("Google API key configured")

In [ ]:
from huggingface_hub import login
login()

## 6. Load CNN Model

In [ ]:
# Check GPU availability
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
MODEL_PATH = f"{DRIVE_BASE_PATH}/best_model.keras"  # Path to your trained model in Drive

# Load the trained model
print(f"Loading model from {MODEL_PATH}...")
try:
    model = tf.keras.models.load_model(MODEL_PATH)
    print("Model loaded successfully")
    print(f"Model input shape: {model.input_shape}")
    print(f"Model output shape: {model.output_shape}")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please ensure the model file exists and is in the correct format or saved with `tf.keras.models.save_model`.")

In [ ]:
# Evaluate CNN Model on Test Dataset
# Note: You need to provide the path to your test dataset

def evaluate_cnn_model(test_data_path, model):
    """
    Evaluate the CNN model on test dataset and display metrics.

    Args:
        test_data_path: Path to test dataset directory or TensorFlow dataset
        model: Loaded Keras model
    """
    from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
    from tensorflow.keras.preprocessing.image import ImageDataGenerator
    import seaborn as sns

    # Create validation data generator (no augmentation, same as training validation)
    val_datagen = ImageDataGenerator(validation_split=0.2)

    validation_generator = val_datagen.flow_from_directory(
        test_data_path,
        target_size=IMG_SIZE,
        batch_size=32,
        class_mode='categorical',
        subset='validation',  # Use the same validation split as training
        shuffle=False,  # Important: Don't shuffle for evaluation
        seed=42  # Same seed as training for consistent split
    )

    print(f"Found {validation_generator.samples} validation images")
    print(f"Classes: {list(validation_generator.class_indices.keys())}")

    # Get predictions
    print("\nGenerating predictions...")
    y_pred_probs = model.predict(validation_generator, verbose=1)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = validation_generator.classes

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')

    print("\n" + "="*60)
    print("CNN MODEL EVALUATION METRICS")
    print("="*60)
    print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print("="*60)

    # Classification report
    print("\nDetailed Classification Report:")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title('Confusion Matrix - Validation Set')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()

    # Per-class accuracy
    print("\nPer-Class Accuracy:")
    print("="*60)
    for i, class_name in enumerate(CLASS_NAMES):
        class_mask = (y_true == i)
        if class_mask.sum() > 0:
            class_acc = (y_pred[class_mask] == i).sum() / class_mask.sum()
            print(f"{class_name:15s}: {class_acc:.4f} ({class_acc*100:.2f}%)")
    print("="*60)

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'confusion_matrix': cm,
        'y_true': y_true,
        'y_pred': y_pred
    }

# Run evaluation on validation set
DATASET_DIR = f"{DRIVE_BASE_PATH}/data/unified_dataset"  # Update path to match your dataset

print("Evaluating model on validation set...")
print(f"Dataset path: {DATASET_DIR}")
print("\nNote: Using the same validation split (20%) as during training with seed=42\n")

# Uncomment the line below to run the evaluation:
metrics = evaluate_cnn_model(DATASET_DIR, model)

## 7. Initialize RAG Pipeline Components

In [ ]:
# Initialize embeddings model
print("Initializing embeddings model...")

embeddings = HuggingFaceEmbeddings(
    model_name="google/embeddinggemma-300m",
    model_kwargs={'device': 'cuda' if tf.config.list_physical_devices('GPU') else 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

print("Embeddings model initialized")

In [ ]:
# Load vector store
def load_vectorstore(chunking_method: str = "recursive") -> Chroma:
    collection_name = {
        "recursive": "dermatology_docs_recursive",
        "agentic": "dermatology_docs_agentic"
    }.get(chunking_method.lower(), "dermatology_docs")

    print(f"Loading vector store: {collection_name}...")

    if not os.path.exists(VECTOR_STORE_PATH):
        raise FileNotFoundError(f"Vector store path not found: {VECTOR_STORE_PATH}")

    vectorstore = Chroma(
        persist_directory=VECTOR_STORE_PATH,
        embedding_function=embeddings,
        collection_name=collection_name
    )

    collection = vectorstore._collection
    doc_count = collection.count()

    print(f"Vector store loaded: {doc_count} documents")
    return vectorstore

vectorstore = load_vectorstore(CHUNKING_METHOD)

In [ ]:
# Load chunks for BM25 indexing
def load_chunks_from_vectorstore(vectorstore: Chroma) -> List[Document]:
    print("Loading chunks from vector store...")

    collection = vectorstore._collection
    all_results = collection.get()

    documents = []
    ids = all_results.get("ids", [])
    metadatas = all_results.get("metadatas", [])
    documents_text = all_results.get("documents", [])

    for doc_id, text, metadata in zip(ids, documents_text, metadatas):
        doc = Document(page_content=text, metadata=metadata or {})
        documents.append(doc)

    print(f"Loaded {len(documents)} chunks")
    return documents

chunks = load_chunks_from_vectorstore(vectorstore)

In [ ]:
# Create BM25 retriever
print("Building BM25 index...")
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = RETRIEVAL_K
print(f"BM25 index built with {len(chunks)} documents")

In [ ]:
# Create semantic retriever
semantic_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": RETRIEVAL_K}
)
print("Semantic retriever created")

In [ ]:
# Create hybrid retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, semantic_retriever],
    weights=HYBRID_WEIGHTS
)
print(f"Hybrid retriever created with weights: BM25={HYBRID_WEIGHTS[0]}, Semantic={HYBRID_WEIGHTS[1]}")

In [ ]:
# Initialize Gemini Pro
print("Initializing Gemini Pro model...")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-pro",
    temperature=0.1,  # Low temperature for factual medical responses
    convert_system_message_to_human=True
)
print("Gemini Pro initialized")

## 8. Create Treatment Recommendation Chain

In [ ]:
# Create prompt template for treatment recommendations
treatment_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a friendly, clinically careful dermatology expert speaking directly to the user.
Your top priorities are: (1) evidence-based accuracy grounded ONLY in the provided context,
(2) clear, supportive guidance in plain language, and (3) safety-first recommendations.

STYLE & VOICE
- Address the user as "you". Sound calm, supportive, and non-judgmental.
- Prefer short paragraphs and bullet points. Use everyday language; define any unavoidable jargon.
- Be concise first; elaborate only if needed.

GROUNDING & CITATIONS
- Use ONLY the "Medical literature context" below. Do not rely on prior knowledge.
- If a point comes from the context, reflect it clearly; if unavailable, explicitly say:
  "This isn’t covered in the provided context."
- Do NOT fabricate drugs, dosages, or guidelines.
- If the context includes source identifiers, cite them inline like [1], [2].

SAFETY & TRIAGE
- Always include: red-flag symptoms and when to seek urgent care vs routine dermatology review.
- Consider common sensitivities: children, pregnancy/breastfeeding, known allergies, immunosuppression—only if mentioned in the context; otherwise say context doesn’t specify.
- If treatments require prescription or supervision, say so.

STRUCTURE YOUR ANSWER AS:
1) What {condition} is (1–2 sentences, plain language)
2) What you can do now (self-care/OTC options from context, with simple steps)
3) Medical treatments (from context): typical first-line → second-line → specialist options
   - Include any condition-specific best practices (application frequency, duration, step-up/step-down if present)
   - Include precautions/contraindications if provided
4) When to seek care
   - Urgent: red flags
   - Routine: if symptoms persist/worsen despite the above or per context
5) What the context does NOT cover (one bullet list; be explicit)
6) Sources (brief inline bracketed refs if available)

TONE CHECK
- Avoid moralizing (“good/bad skin”). Reassure that many conditions are manageable.
- Encourage adherence and realistic expectations (timelines if provided in context).

If information is missing, do NOT guess. Be transparent about gaps.

When citing evidence from the provided medical literature, include inline APA-style citations
with:
- Author or organization name
- Year (if available; else 'n.d.')
- Title (shortened if very long)

Example:
"Moisturizers are recommended for mild eczema (Thomson Team, 2024, *3 Most Common Skin Disorders in Singapore*)."

If the context includes multiple relevant sources, you may cite up to two inline, separated by semicolons.

"""),
    ("human", """You’ve just been told you have {condition}.
Using ONLY the medical literature context below, talk directly to me and answer:

- Keep it practical and kind.
- Follow the required structure and rules above.

Medical literature context:
{context}

What are the treatment options and guidelines for my {condition}?""")
])

# Helper function to format documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Create RAG chain
treatment_rag_chain = (
    {
        "context": hybrid_retriever | format_docs,
        "condition": RunnablePassthrough()
    }
    | treatment_prompt
    | llm
    | StrOutputParser()
)

print("Treatment recommendation chain created")

## 9. Image Classification Function

In [ ]:
def classify_skin_condition(img_path: str, display_image: bool = True) -> tuple:
    """
    Classify a skin condition from an image.

    Args:
        img_path: Path to the image file
        display_image: Whether to display the image

    Returns:
        Tuple of (predicted_class, confidence, all_probabilities)
    """
    # Load and preprocess image
    img = image.load_img(img_path, target_size=IMG_SIZE)

    # Display image if requested
    if display_image:
        plt.figure(figsize=(6, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.title('Input Image')
        plt.show()

    # Convert to array and preprocess
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)

    # Make prediction
    predictions = model.predict(img_array, verbose=0)
    predicted_class_idx = np.argmax(predictions[0])
    predicted_class = CLASS_NAMES[predicted_class_idx]
    confidence = predictions[0][predicted_class_idx]

    # Print results
    print("="*80)
    print("Classification Results:")
    print("="*80)
    print(f"Predicted Condition: {predicted_class}")
    print(f"Confidence: {confidence*100:.2f}%")
    print("\nAll Class Probabilities:")
    for i, class_name in enumerate(CLASS_NAMES):
        print(f"  {class_name}: {predictions[0][i]*100:.2f}%")
    print("="*80)

    return predicted_class, confidence, predictions[0]

print("Classification function defined")

## 10. Treatment Recommendation Function

In [ ]:
def get_treatment_recommendations(condition: str, show_sources: bool = True) -> dict:
    """
    Get treatment recommendations for a diagnosed skin condition.

    Args:
        condition: The diagnosed skin condition
        show_sources: Whether to display source documents

    Returns:
        Dictionary with treatment recommendations and sources
    """
    print("\nRetrieving treatment recommendations...")
    print("="*80)

    # Get recommendations
    recommendations = treatment_rag_chain.invoke(condition)

    print("Treatment Recommendations:")
    print("="*80)
    print(recommendations)
    print("="*80)

    # Get source documents
    source_docs = None
    if show_sources:
        # Create a query for the condition
        query = f"treatment options and guidelines for {condition}"
        source_docs = hybrid_retriever.invoke(query)

        print("\nSource Documents:")
        print("="*80)
        for i, doc in enumerate(source_docs, 1):
            category = doc.metadata.get("category", "Unknown")
            source_file = doc.metadata.get("source_file", "Unknown")
            page = doc.metadata.get("page", "?")
            print(f"\n[{i}] Category: {category} | File: {source_file} | Page: {page}")
            print(f"    Preview: {doc.page_content[:200]}...")
        print("="*80)

    return {
        "condition": condition,
        "recommendations": recommendations,
        "sources": source_docs
    }

print("Treatment recommendation function defined")

## 11. Integrated Pipeline Function

In [ ]:
def diagnose_and_recommend(img_path: str,
                          display_image: bool = True,
                          show_sources: bool = True,
                          confidence_threshold: float = 0.5) -> dict:
    """
    Complete pipeline: Classify skin condition and provide treatment recommendations.

    Args:
        img_path: Path to the skin condition image
        display_image: Whether to display the input image
        show_sources: Whether to display source documents
        confidence_threshold: Minimum confidence for providing recommendations

    Returns:
        Dictionary containing classification and treatment information
    """
    print("\n" + "#"*80)
    print("#" + " "*78 + "#")
    print("#" + " INTEGRATED SKIN CONDITION ANALYSIS AND TREATMENT SYSTEM ".center(78) + "#")
    print("#" + " "*78 + "#")
    print("#"*80 + "\n")

    # Step 1: Classify the skin condition
    print("STEP 1: SKIN CONDITION CLASSIFICATION")
    predicted_class, confidence, probabilities = classify_skin_condition(
        img_path,
        display_image=display_image
    )

    # Check confidence threshold
    if confidence < confidence_threshold:
        print("\n" + "!"*80)
        print(f"WARNING: Low confidence ({confidence*100:.2f}%). Results may not be reliable.")
        print("Please consult a healthcare professional for accurate diagnosis.")
        print("!"*80)

    # Step 2: Get treatment recommendations
    print("\n" + "="*80)
    print("STEP 2: TREATMENT RECOMMENDATIONS")
    print("="*80)

    treatment_info = get_treatment_recommendations(
        predicted_class,
        show_sources=show_sources
    )

    # Compile results
    results = {
        "image_path": img_path,
        "classification": {
            "predicted_condition": predicted_class,
            "confidence": float(confidence),
            "all_probabilities": {CLASS_NAMES[i]: float(probabilities[i])
                                 for i in range(len(CLASS_NAMES))}
        },
        "treatment": treatment_info
    }

    print("\n" + "#"*80)
    print("#" + " ANALYSIS COMPLETE ".center(78) + "#")
    print("#"*80 + "\n")

    return results

print("Integrated pipeline function defined")

## 12. Usage Example

Upload an image and run the complete pipeline:

In [ ]:
# Example: Upload and analyze an image
# Replace 'path/to/your/image.jpg' with the actual path to your test image

IMAGE_PATH = "path/to/your/image.jpg"  # Update this path

# Run the integrated pipeline
results = diagnose_and_recommend(
    img_path=IMAGE_PATH,
    display_image=True,
    show_sources=True,
    confidence_threshold=0.5
)

## 13. Interactive Image Upload (For Jupyter/Colab)

Use this section for interactive image upload in Jupyter or Google Colab:

In [ ]:
# For Google Colab
try:
    from google.colab import files

    def upload_and_diagnose():
        print("Please upload an image file...")
        uploaded = files.upload()

        for filename in uploaded.keys():
            print(f"\nProcessing uploaded file: {filename}")
            result = diagnose_and_recommend(
                filename,
                display_image=True,
                show_sources=True
            )
            return result

    print("Google Colab upload function ready")
    print("Run: result = upload_and_diagnose()")

except ImportError:
    print("Not running in Google Colab - using standard file upload")

## 14. Gradio Interface

Interactive web interface for skin condition analysis


In [ ]:
# Install Gradio
%pip install gradio

In [ ]:
import gradio as gr
print("Gradio imported successfully")

In [ ]:
def create_gradio_output(results: dict) -> str:
    """
    Create beautifully formatted HTML output for Gradio interface.

    Args:
        results: Dictionary containing classification and treatment information

    Returns:
        HTML formatted string for display
    """
    condition = results['classification']['predicted_condition']
    confidence = results['classification']['confidence'] * 100
    recommendations = results['treatment']['recommendations']

    # Create confidence color based on value
    if confidence >= 70:
        confidence_color = "#10b981"  # Green
        confidence_label = "High Confidence"
    elif confidence >= 50:
        confidence_color = "#f59e0b"  # Orange
        confidence_label = "Moderate Confidence"
    else:
        confidence_color = "#ef4444"  # Red
        confidence_label = "Low Confidence"

    # Format recommendations text (convert markdown-like formatting to HTML)
    recommendations_html = recommendations.replace('\n\n', '</p><p style="margin: 12px 0;">') \
                                         .replace('\n', '<br>') \
                                         .replace('###', '<strong>') \
                                         .replace('**', '</strong>')

            # Build complete HTML output with dark theme
    html_output = f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif;
                max-width: 100%; margin: 0 auto; padding: 0;">

        <!-- Main Diagnosis Card -->
        <div style="background: #1e293b; border-radius: 12px; padding: 30px;
                    box-shadow: 0 4px 6px rgba(0, 0, 0, 0.3); margin-bottom: 20px;
                    border: 2px solid {confidence_color};">
            <div style="text-align: center; margin-bottom: 20px;">
                <div style="display: inline-block; padding: 10px 24px;
                            background-color: {confidence_color}; color: white;
                            border-radius: 20px; font-size: 14px; font-weight: 700; margin-bottom: 16px;
                            box-shadow: 0 2px 4px rgba(0, 0, 0, 0.3);">
                    {confidence_label}
                </div>
                <h2 style="margin: 8px 0; font-size: 36px; font-weight: 700;
                           background: linear-gradient(135deg, #a78bfa 0%, #c084fc 100%);
                           -webkit-background-clip: text;
                           -webkit-text-fill-color: transparent;
                           background-clip: text;">
                    {condition}
                </h2>
            </div>

            <!-- Confidence Warning -->
            {"<div style='background: linear-gradient(135deg, #fbbf24 0%, #f59e0b 100%); border-left: 4px solid #d97706; padding: 16px; border-radius: 8px; margin-top: 20px; box-shadow: 0 2px 4px rgba(0, 0, 0, 0.3);'><p style='margin: 0; color: #1e293b; font-weight: 700;'>Warning</p><p style='margin: 8px 0 0 0; color: #1e293b; font-weight: 500;'>This result has " + confidence_label.lower() + ". Please consult a healthcare professional for accurate diagnosis.</p></div>" if confidence < 70 else ""}
        </div>

        <!-- Treatment Recommendations Card -->
        <div style="background: #1e293b; border-radius: 12px; padding: 30px;
                    box-shadow: 0 4px 6px rgba(0, 0, 0, 0.3); margin-bottom: 20px;
                    border: 1px solid #334155;">
            <h3 style="margin: 0 0 20px 0; font-size: 22px; font-weight: 700; color: #a78bfa;">
                Treatment Recommendations
            </h3>
            <div style="color: #cbd5e1; line-height: 1.8; font-size: 15px;">
                <p style="margin: 12px 0;">{recommendations_html}</p>
            </div>
        </div>

        <!-- Disclaimer -->
        <div style="background: linear-gradient(135deg, #fbbf24 0%, #f59e0b 100%);
                    border-radius: 12px; padding: 20px;
                    border: 1px solid #d97706; text-align: center;
                    box-shadow: 0 4px 6px rgba(0, 0, 0, 0.3);">
            <p style="margin: 0; color: #1e293b; font-size: 14px; line-height: 1.6; font-weight: 500;">
                <strong style="color: #1e293b; font-weight: 700;">Medical Disclaimer:</strong> This tool is for educational purposes only and does not replace
                professional medical advice, diagnosis, or treatment. Always seek the advice of your physician or
                other qualified health provider with any questions you may have regarding a medical condition.
            </p>
        </div>
    </div>
    """

    return html_output

print("Gradio output formatter created")


In [ ]:
def gradio_predict(image):
    """
    Gradio interface function for skin condition prediction.

    Args:
        image: PIL Image or numpy array from Gradio

    Returns:
        HTML formatted output string
    """
    try:
        # Save the image temporarily
        temp_image_path = "temp_upload.jpg"

        # Convert to PIL Image if needed
        if isinstance(image, np.ndarray):
            image = Image.fromarray(image)

        # Save image
        image.save(temp_image_path)

        # Run the diagnosis and recommendation pipeline
        # Set display_image=False since we're in Gradio
        results = diagnose_and_recommend(
            img_path=temp_image_path,
            display_image=False,
            show_sources=False,  # Don't print sources in console
            confidence_threshold=0.0  # Process all images
        )

        # Create formatted output
        html_output = create_gradio_output(results)

        return html_output

    except Exception as e:
        error_html = f"""
        <div style="padding: 30px; background: #1e293b;
                    border-radius: 12px; border: 2px solid #ef4444; color: #fca5a5;
                    box-shadow: 0 4px 6px rgba(0, 0, 0, 0.3);">
            <h3 style="margin: 0 0 12px 0; color: #f87171; font-weight: 700;">Error Processing Image</h3>
            <p style="margin: 0; color: #fca5a5;"><strong>Error:</strong> {str(e)}</p>
            <p style="margin: 12px 0 0 0; font-size: 14px; color: #cbd5e1;">
                Please try again with a different image or check that all models are loaded correctly.
            </p>
        </div>
        """
        return error_html

print("Gradio prediction function created")


In [ ]:
# Create Gradio Interface
def create_gradio_interface():
    """
    Create and configure the Gradio interface.
    """

    # Custom CSS for dark theme
    custom_css = """
    .gradio-container {
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif !important;
        background-color: #0f172a !important;
    }
    .header-text {
        text-align: center;
        padding: 20px;
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        color: white;
        border-radius: 12px;
        margin-bottom: 20px;
        box-shadow: 0 4px 6px rgba(0, 0, 0, 0.3);
    }
    .info-box {
        background: #1e293b;
        border-radius: 12px;
        padding: 24px;
        box-shadow: 0 4px 6px rgba(0, 0, 0, 0.3);
        border: 1px solid #334155;
        height: 100%;
    }
    .upload-box {
        margin-bottom: 4px;
        border: 2px dashed #667eea !important;
        border-radius: 12px !important;
        background: #1e293b !important;
    }
    .upload-container {
        background: #1e293b;
        border-radius: 12px;
        padding: 24px;
        box-shadow: 0 4px 6px rgba(0, 0, 0, 0.3);
        border: 1px solid #334155;
    }
    button {
        border: none !important;
        box-shadow: 0 2px 4px rgba(0, 0, 0, 0.3) !important;
    }
    .section-title {
        color: #a78bfa;
        font-size: 20px;
        font-weight: 700;
        margin-bottom: 16px;
    }
    label {
        color: #e2e8f0 !important;
    }
    """

        # Example images info text
    info_content = """
    <div style="color: #e2e8f0; line-height: 1.8;">
        <h3 style="color: #a78bfa; margin-top: 0; font-weight: 700;">Supported Conditions</h3>
        <p style="margin: 12px 0; color: #cbd5e1;">Our AI model can identify the following skin conditions:</p>
        <ul style="margin: 8px 0; padding-left: 20px; color: #cbd5e1;">
            <li><strong style="color: #e2e8f0;">Acne</strong>: Inflammatory skin condition with pimples and blemishes</li>
            <li><strong style="color: #e2e8f0;">Eczema</strong>: Itchy, inflamed skin patches</li>
            <li><strong style="color: #e2e8f0;">Psoriasis</strong>: Scaly, red skin patches</li>
            <li><strong style="color: #e2e8f0;">Rosacea</strong>: Facial redness and visible blood vessels</li>
            <li><strong style="color: #e2e8f0;">Urticaria</strong>: Hives or welts on the skin</li>
            <li><strong style="color: #e2e8f0;">Vitiligo</strong>: Loss of skin pigmentation</li>
        </ul>

        <h3 style="color: #a78bfa; margin-top: 24px; font-weight: 700;">How to Use</h3>
        <ol style="margin: 8px 0; padding-left: 20px; color: #cbd5e1;">
            <li>Upload a clear image of the skin condition</li>
            <li>Click the "Analyze Image" button</li>
            <li>Wait for the AI to process the image</li>
            <li>Review the diagnosis and treatment recommendations below</li>
            <li><strong style="color: #e2e8f0;">Always consult a healthcare professional for proper diagnosis</strong></li>
        </ol>
    </div>
    """

    # Create the interface using Blocks for custom layout
    with gr.Blocks(css=custom_css, title="Skin Condition Analyzer", theme=gr.themes.Soft(primary_hue="violet", neutral_hue="slate").set(
        body_background_fill="#0f172a",
        body_background_fill_dark="#0f172a",
        block_background_fill="#1e293b",
        block_background_fill_dark="#1e293b",
        input_background_fill="#1e293b",
        input_background_fill_dark="#1e293b",
    )) as demo:

        # Header
        gr.Markdown(
            """
            <div class="header-text">
                <h1 style="margin: 0; font-size: 36px;">AI Skin Condition Analyzer</h1>
                <p style="margin: 10px 0 0 0; font-size: 18px; opacity: 0.9;">
                    Advanced dermatology diagnosis and treatment recommendation system
                </p>
            </div>
            """
        )

        # Top row - Information (left) and Upload (right)
        with gr.Row():
            # Left column - Information
            with gr.Column(scale=1):
                gr.Markdown('<h3 class="section-title">Description</h3>')
                gr.HTML(f'<div class="info-box">{info_content}</div>')

            # Right column - Upload
            with gr.Column(scale=1):
                gr.Markdown('<h3 class="section-title">Upload Skin Image</h3>')

                image_input = gr.Image(
                    label="Select Image",
                    type="pil",
                    height=400,
                    sources=["upload"],
                    elem_classes=["upload-box"]
                )

                analyze_btn = gr.Button(
                    "Analyze Image",
                    variant="primary",
                    size="lg"
                )
                gr.HTML('</div>')

                # Bottom row - Results (full width)
        with gr.Row():
            with gr.Column():
                gr.Markdown('<h3 class="section-title" style="margin-top: 20px;">Analysis Results</h3>')

                output_html = gr.HTML(
                    label="Results",
                    value="""
                    <div style="padding: 40px; text-align: center;
                                background: #1e293b; border-radius: 12px;
                                border: 2px dashed #334155; color: #94a3b8;
                                box-shadow: 0 4px 6px rgba(0, 0, 0, 0.3);">
                        <h3 style="margin: 16px 0 8px 0; color: #a78bfa; font-weight: 700;">Ready to Analyze</h3>
                        <p style="margin: 0; color: #cbd5e1;">Upload an image and click "Analyze Image" to get started</p>
                    </div>
                    """
                )

                # Footer with disclaimer
        gr.Markdown(
            """
            ---

            <div style="text-align: center; padding: 20px; background: linear-gradient(135deg, #fbbf24 0%, #f59e0b 100%);
                        border-radius: 12px; border: 1px solid #d97706; margin-top: 20px; box-shadow: 0 4px 6px rgba(0, 0, 0, 0.3);">
                <p style="margin: 0; color: #1e293b; font-weight: 700; font-size: 16px;">
                    Important Medical Disclaimer
                </p>
                <p style="margin: 8px 0 0 0; color: #1e293b; line-height: 1.6; font-weight: 500;">
                    This AI tool is designed for educational and informational purposes only.
                    It does NOT replace professional medical advice, diagnosis, or treatment.
                    Always consult a qualified healthcare provider for proper diagnosis and treatment of any skin condition.
                    Never disregard professional medical advice or delay seeking it because of information from this tool.
                </p>
            </div>

            <div style="text-align: center; margin-top: 20px; color: #94a3b8; font-size: 14px;">
                <p style="margin: 4px 0;">Powered by EfficientNetV2M (CNN) + RAG Pipeline with Gemini Pro</p>
                <p style="margin: 4px 0;">© 2025 Skin Condition Analyzer | Built with TensorFlow, LangChain & Gradio</p>
            </div>
            """
        )

        # Connect the button to the prediction function
        analyze_btn.click(
            fn=gradio_predict,
            inputs=image_input,
            outputs=output_html,
            api_name="predict"
        )

    return demo

print("Gradio interface created")


In [ ]:
# Launch the Gradio interface
demo = create_gradio_interface()

# Launch with public sharing enabled (for Colab)
# Set share=False if you don't want a public link
demo.launch(
    share=True,  # Set to True to get a public shareable link
    debug=True,  # Enable debug mode for better error messages
    show_error=True  # Show detailed error messages
)

print("\n" + "="*80)
print("Gradio interface launched successfully!")
print("="*80)
print("\nThe interface should open automatically.")
print("If running in Colab, you'll see a public URL above.")
print("If running locally, the interface will open in your browser.")
print("\nTip: You can share the public URL with others to let them use your app!")
print("="*80)
